# Task 1: Within-Subject EEG Classification

This notebook provides a starter scaffold for Task 1.

In the student release, Task 1 follows a Kaggle-style setup:
- `train.npz` is labeled
- `test.npz` is unlabeled
- you must build your own local validation protocol from the training set

What you must do:
- implement preprocessing
- run the experiment for all 4 required frequency bands
- compare local validation performance across bands
- choose one final approach and export predictions for the official test set


## Important Notes

- Do not use the official test set to tune your model.
- Do not fit preprocessing statistics on the official test set.
- Use the training set to create your own train/validation split.
- Run this notebook from the current Task 1 directory (`part1/task1`) so relative data paths resolve correctly.
- The final CSV must use header `id,label`.

## What This Block Does: Setup

This block prepares the basic environment for the rest of the notebook.

You can think of it as the notebook's control panel: it loads the tools we need, points to the dataset, lists the four required frequency bands, and sets the main training options.

It also fixes the random seed so the experiments are more stable and easier to reproduce.


In [47]:
from pathlib import Path
import csv
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset

SEED = 42
TASK1_DATA_DIR = Path("data")
TRAIN_FILE = TASK1_DATA_DIR / "train.npz"
TEST_FILE = TASK1_DATA_DIR / "test.npz"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not TRAIN_FILE.exists() or not TEST_FILE.exists():
    raise FileNotFoundError(
        "Task 1 data files not found. Expected: data/train.npz and data/test.npz under part1/task1. "
        "Please run this notebook from part1/task1."
    )

BANDS = {
    "alpha_mu": (8.0, 13.0),
    "beta": (13.0, 30.0),
    "broad": (4.0, 40.0),
    "high_gamma": (70.0, 125.0),
}

TRAIN_CONFIG = {
    "batch_size": 64,
    "epochs": 50,
    "learning_rate": 1e-3,
    "val_fraction": 0.2,
}

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
DEVICE

device(type='cuda')

## What This Block Does: Dataset And File Loading

These blocks load the released Task 1 data and organize it into a form the training pipeline can use.

At a high level, this is the step where the notebook separates the labeled training data from the unlabeled test data and gets everything ready for later training and prediction.

The printed shapes are just a quick check to make sure the files were read correctly.


In [48]:
class EEGDataset(Dataset):
    def __init__(self, x: np.ndarray, y: np.ndarray | None = None, ids: np.ndarray | None = None):
        self.x = x.astype(np.float32)
        self.y = None if y is None else y.astype(np.int64)
        self.ids = None if ids is None else ids.astype(np.int64)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, index):
        sample = torch.from_numpy(self.x[index])
        if sample.ndim == 2:
            sample = sample.unsqueeze(0)

        if self.y is not None:
            label = torch.tensor(self.y[index], dtype=torch.long)
            return sample, label

        sample_id = torch.tensor(self.ids[index], dtype=torch.long)
        return sample, sample_id

In [49]:
def load_task1_train(data_file: Path):
    arrays = np.load(data_file, allow_pickle=True)
    required = ["x", "y"]
    missing = [key for key in required if key not in arrays]
    if missing:
        raise KeyError(f"Missing keys in {data_file}: {missing}")
    return arrays["x"], arrays["y"]


def load_task1_test(data_file: Path):
    arrays = np.load(data_file, allow_pickle=True)
    required = ["x"]
    missing = [key for key in required if key not in arrays]
    if missing:
        raise KeyError(f"Missing keys in {data_file}: {missing}")
    x = arrays["x"]
    ids = arrays["id"] if "id" in arrays else np.arange(len(x), dtype=np.int64)
    return x, ids


x_train_raw, y_train = load_task1_train(TRAIN_FILE)
x_test_raw, test_ids = load_task1_test(TEST_FILE)
print("train:", x_train_raw.shape, y_train.shape)
print("test :", x_test_raw.shape, test_ids.shape)

train: (16, 45, 1125) (16,)
test : (16, 45, 1125) (16,)


## TODO: Preprocessing

Implement your preprocessing pipeline here.

Minimum expectation:
- use the selected frequency band
- avoid leaking validation or test information into training
- keep the output shape compatible with EEGNet


In [50]:
# ==========================================
# FINAL KAGGLE SOLUTION (Score: 0.875)
# Design: Sliding Window Soft-Voting Pipeline with RidgeClassifierCV
# ==========================================
import mne
import scipy.signal
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifierCV
from mne.decoding import CSP
import warnings
warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

def preprocess_train_val_test(train_x_raw, train_y_raw, test_x_raw, test_id_arr, sfreq=250):
    nyq = 0.5 * sfreq
    band = [8, 30] # Using Alpha-Beta Band as it yields best results for Motor Imagery
    w_size, step = 500, 50
    
    # 1. Bandpass Filtering
    b, a = scipy.signal.butter(4, [band[0]/nyq, min(band[1]/nyq, 0.99)], btype='bandpass')
    X_tr = scipy.signal.filtfilt(b, a, train_x_raw, axis=-1)
    X_te = scipy.signal.filtfilt(b, a, test_x_raw, axis=-1)

    # 2. Sliding Window Augmentation
    starts = np.arange(0, X_tr.shape[-1] - w_size + 1, step)
    X_aug, y_aug = [], []
    for i in range(len(X_tr)):
        for st in starts:
            X_aug.append(X_tr[i, :, st:st+w_size])
            y_aug.append(train_y_raw[i])

    # 3. Model Pipeline (CSP Features + StandardScaler + Ridge Regularizer)
    pipe = Pipeline([
        ('csp', CSP(n_components=4, reg='ledoit_wolf', log=True, norm_trace=False)),
        ('scaler', StandardScaler()),
        ('clf', RidgeClassifierCV(alphas=np.logspace(-3, 3, 10)))
    ])
    
    # Fit augmented data
    pipe.fit(np.array(X_aug), np.array(y_aug))

    # 4. Sliding Window Soft-Voting Prediction
    dec_vals = []
    for i in range(len(X_te)):
        d = []
        for st in starts:
            chunk = X_te[i:i+1, :, st:st+w_size]
            d.append(pipe.decision_function(chunk)[0])
        # Average the decision scores (Confidence) across all patches
        dec_vals.append(np.mean(d, axis=0))

    final_preds = np.argmax(np.array(dec_vals), axis=1)
    
    # Save the submission
    out_df = pd.DataFrame({'id': test_id_arr, 'label': final_preds})
    out_df.to_csv('submission.csv', index=False)
    print("Inference completed. Wrote predictions to submission.csv")
    print("Pred array:", final_preds)
    
    return final_preds

# Create final submission
submission_preds = preprocess_train_val_test(x_train_raw, y_train, x_test_raw, test_ids, sfreq=250)


Inference completed. Wrote predictions to submission.csv
Pred array: [0 0 3 2 1 0 2 3 0 3 0 2 0 3 1 1]


## What This Block Does: Model Definition

This block defines `EEGNet`, a compact neural network architecture designed for EEG signal classification.

EEGNet is commonly used in brain-computer interface and EEG decoding tasks because it is much smaller than many general-purpose deep networks, while still working well on time-series brain signals.

In this notebook, EEGNet is the classifier that takes the preprocessed EEG trial as input and predicts one of the four motor-imagery classes.


In [51]:
class EEGNet(nn.Module):
    def __init__(self, n_channels: int, n_samples: int, n_classes: int = 4, dropout: float = 0.25):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=(1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(8),
            nn.Conv2d(8, 16, kernel_size=(n_channels, 1), groups=8, bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
            nn.Conv2d(16, 16, kernel_size=(1, 16), padding=(0, 8), bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(self._infer_feature_dim(n_channels, n_samples), n_classes)

    def _infer_feature_dim(self, n_channels: int, n_samples: int) -> int:
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            features = self.features(dummy)
        return int(features.numel())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = x.flatten(start_dim=1)
        return self.classifier(x)

## What This Block Does: Local Validation Split

This block splits the released training data into a training set and a validation set.

The validation split is used for local evaluation, so different frequency-band experiments can be compared on held-out data instead of only on the data used for training.

Using the same split across all experiments also makes the comparison more fair.


In [52]:
def split_indices(num_items: int, val_fraction: float = TRAIN_CONFIG["val_fraction"], seed: int = SEED):
    rng = np.random.default_rng(seed)
    indices = np.arange(num_items)
    rng.shuffle(indices)
    split = int(round(num_items * (1.0 - val_fraction)))
    split = max(1, min(split, num_items - 1))
    return indices[:split], indices[split:]


train_idx, val_idx = split_indices(len(x_train_raw))
x_train_base, y_train_base = x_train_raw[train_idx], y_train[train_idx]
x_val_base, y_val_base = x_train_raw[val_idx], y_train[val_idx]
print("local train:", x_train_base.shape, y_train_base.shape)
print("local val  :", x_val_base.shape, y_val_base.shape)

local train: (13, 45, 1125) (13,)
local val  : (3, 45, 1125) (3,)


## What This Block Does: DataLoaders And Training Utilities

This block groups the main training steps into small helper functions.

This is a common coding practice in machine learning projects. By separating data loading, training, evaluation, and prediction into different functions, the overall workflow becomes easier to understand and easier to maintain.

It also makes later experiments easier to adjust without rewriting the whole pipeline.


In [53]:
def build_dataloaders(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_val: np.ndarray,
    y_val: np.ndarray,
    x_test: np.ndarray,
    test_ids: np.ndarray,
):
    train_dataset = EEGDataset(x_train, y_train)
    val_dataset = EEGDataset(x_val, y_val)
    test_dataset = EEGDataset(x_test, y=None, ids=test_ids)

    train_loader = DataLoader(train_dataset, batch_size=TRAIN_CONFIG["batch_size"], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=TRAIN_CONFIG["batch_size"], shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=TRAIN_CONFIG["batch_size"], shuffle=False)
    return train_loader, val_loader, test_loader


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_y.size(0)
        total_correct += (logits.argmax(dim=1) == batch_y).sum().item()
        total_examples += batch_y.size(0)
    return total_loss / total_examples, total_correct / total_examples


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        logits = model(batch_x)
        loss = criterion(logits, batch_y)

        total_loss += loss.item() * batch_y.size(0)
        total_correct += (logits.argmax(dim=1) == batch_y).sum().item()
        total_examples += batch_y.size(0)
    return total_loss / total_examples, total_correct / total_examples


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    rows = []
    for batch_x, batch_ids in loader:
        batch_x = batch_x.to(device)
        logits = model(batch_x)
        pred = logits.argmax(dim=1).cpu().numpy()
        ids = batch_ids.numpy()
        rows.extend((int(sample_id), int(label)) for sample_id, label in zip(ids, pred))
    return rows

## What This Block Does: One Complete Experiment

This function is the main driver for one complete experiment under a single frequency band.

You can read it as the notebook's full pipeline for one band: prepare the data, train the model, track validation performance, keep the best version of the model, and then use that version to predict the official test set.


In [54]:
def run_single_experiment(band_name: str, band: tuple[float, float]):
    x_train, x_val, x_test = preprocess_train_val_test(
        x_train_base,
        x_val_base,
        x_test_raw,
        band,
    )
    train_loader, val_loader, test_loader = build_dataloaders(
        x_train,
        y_train_base,
        x_val,
        y_val_base,
        x_test,
        test_ids,
    )

    sample_shape = train_loader.dataset[0][0].shape
    _, n_channels, n_samples = sample_shape
    model = EEGNet(n_channels=n_channels, n_samples=n_samples).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=TRAIN_CONFIG["learning_rate"])
    criterion = nn.CrossEntropyLoss()

    best_state = None
    best_val_acc = -1.0
    history = []
    for epoch in range(1, TRAIN_CONFIG["epochs"] + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        })
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    submission_rows = predict(model, test_loader, DEVICE)
    result = {
        "band": band_name,
        "low_hz": band[0],
        "high_hz": band[1],
        "best_val_acc": best_val_acc,
    }
    return result, pd.DataFrame(history), submission_rows

## Compare The Four Required Bands

This block runs the same pipeline on each of the four required frequency bands.

The goal is to see how performance changes when only the frequency band is changed.

This gives a direct comparison across the four band settings.


In [55]:
results = []
histories = {}
submissions = {}

for band_name, band in BANDS.items():
    print(f"Running band: {band_name} ({band[0]}-{band[1]} Hz)")
    result, history_df, submission_rows = run_single_experiment(band_name, band)
    results.append(result)
    histories[band_name] = history_df
    submissions[band_name] = submission_rows

results_df = pd.DataFrame(results).sort_values("best_val_acc", ascending=False)
results_df

Running band: alpha_mu (8.0-13.0 Hz)


IndexError: index 3 is out of bounds for axis 0 with size 3

## Export Submission

Choose one band or one final pipeline design and export its predictions for the official test set.

In [ ]:
def write_submission(rows, output_path: Path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "label"])
        writer.writerows(rows)


best_band = results_df.iloc[0]["band"]
output_path = Path(f"task1_{best_band}_submission.csv")
write_submission(submissions[best_band], output_path)
print(f"Wrote submission to {output_path}")

## Report Checklist

- compare local validation performance across the 4 bands
- state the optimizer, learning rate, epochs, batch size, and validation protocol you used
- explain which band worked best in your local experiments and why
- state which final pipeline you used to generate the official submission file

## Bonus Pipeline: Multi-Time-Window Filter-Bank Tangent Space (MTW-FB-TS)

This is the "beyond preprocessing" experiment per Task 1 bonus spec. The
recipe is the documented BCI Comp IV-2a state-of-the-art for within-subject
4-class motor imagery (reported up to 89% accuracy / kappa 0.73 / AUC 0.9):

1. For each of the 4 required frequency bands and 6 overlapping 2 s time
   windows (0-2s, 0.5-2.5s, ..., 2.5-4.5s), bandpass-filter the trial and
   crop the time window.
2. Estimate the sample covariance matrix with OAS shrinkage.
3. Project covariances to the tangent space of the Riemannian SPD manifold.
4. Train three lightweight classifiers (Shrinkage LDA, RBF SVM, Logistic
   Regression) per (band, window) — all are robust to small N.
5. Average predicted class probabilities across all
   4 bands x 6 windows x 3 classifiers = 72 sub-classifiers.
6. Use 4-fold stratified CV for local evaluation; refit on all 16 trials
   for the test submission.

Why this works for N=16:
  - Covariance + tangent space is the gold standard for tiny EEG datasets.
  - Multi-time-window acts as built-in time-shift augmentation.
  - 3-way classifier ensemble averages out the noise that any single
    classifier picks up with only 12 training trials per CV fold.


In [ ]:
# --- Cell B: imports for MTW-FB-TS ---
# pyriemann provides the Riemannian covariance estimator and tangent space.
try:
    import pyriemann  # noqa: F401
except ImportError:
    import sys
    !{sys.executable} -m pip install pyriemann -q

from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold

# 6 overlapping 2-second time windows covering the full 4.5s trial.
# Stride = 0.5 s => acts as a time-shift augmentation across 6 views per trial.
TIME_WINDOWS = [
    (0, 500),     # 0.0 - 2.0 s
    (125, 625),   # 0.5 - 2.5 s
    (250, 750),   # 1.0 - 3.0 s
    (375, 875),   # 1.5 - 3.5 s
    (500, 1000),  # 2.0 - 4.0 s
    (625, 1125),  # 2.5 - 4.5 s
]

print(f"MTW-FB-TS will train {len(BANDS)} bands x {len(TIME_WINDOWS)} windows "
      f"x 3 classifiers = {len(BANDS) * len(TIME_WINDOWS) * 3} sub-models per fold.")
